# Task 1: Exploratory Data Analysis - Financial News Dataset

This notebook performs comprehensive exploratory data analysis on the financial news dataset to understand patterns, trends, and statistical properties.

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import re
from collections import Counter
from wordcloud import WordCloud
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("Libraries imported successfully")

In [ ]:
# Load the dataset
df = pd.read_csv('../data/raw/raw_analyst_ratings.csv')

# Display basic information
print(f"Dataset shape: {df.shape}")
print("\nColumn names:")
print(df.columns.tolist())
print("\nFirst few rows:")
print(df.head())

In [ ]:
# Data preprocessing
# Convert date column to datetime
df['date'] = pd.to_datetime(df['date'])

# Extract date components
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day
df['hour'] = df['date'].dt.hour
df['day_of_week'] = df['date'].dt.day_name()

# Calculate headline length
df['headline_length'] = df['headline'].str.len()
df['headline_word_count'] = df['headline'].str.split().str.len()

print("Data preprocessing completed")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")

## 1. Descriptive Statistics

In [ ]:
# Basic statistics for headline lengths
print("Headline Length Statistics:")
print(df['headline_length'].describe())

print("\nHeadline Word Count Statistics:")
print(df['headline_word_count'].describe())

In [ ]:
# Visualization of headline length distribution
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Headline character count distribution
axes[0,0].hist(df['headline_length'], bins=50, alpha=0.7, color='skyblue')
axes[0,0].set_title('Distribution of Headline Character Count')
axes[0,0].set_xlabel('Character Count')
axes[0,0].set_ylabel('Frequency')
axes[0,0].axvline(df['headline_length'].mean(), color='red', linestyle='--', label=f'Mean: {df["headline_length"].mean():.1f}')
axes[0,0].legend()

# Headline word count distribution
axes[0,1].hist(df['headline_word_count'], bins=30, alpha=0.7, color='lightcoral')
axes[0,1].set_title('Distribution of Headline Word Count')
axes[0,1].set_xlabel('Word Count')
axes[0,1].set_ylabel('Frequency')
axes[0,1].axvline(df['headline_word_count'].mean(), color='red', linestyle='--', label=f'Mean: {df["headline_word_count"].mean():.1f}')
axes[0,1].legend()

# Box plot for headline length
axes[1,0].boxplot(df['headline_length'])
axes[1,0].set_title('Box Plot of Headline Character Count')
axes[1,0].set_ylabel('Character Count')

# Box plot for word count
axes[1,1].boxplot(df['headline_word_count'])
axes[1,1].set_title('Box Plot of Headline Word Count')
axes[1,1].set_ylabel('Word Count')

plt.tight_layout()
plt.show()

## 2. Publisher Analysis

In [ ]:
# Count articles per publisher
publisher_counts = df['publisher'].value_counts().head(20)

print(f"Total unique publishers: {df['publisher'].nunique()}")
print("\nTop 20 publishers by article count:")
print(publisher_counts)

In [ ]:
# Extract email domains from publisher names
def extract_domain(publisher):
    if '@' in str(publisher):
        return publisher.split('@')[1]
    return publisher

df['publisher_domain'] = df['publisher'].apply(extract_domain)
domain_counts = df['publisher_domain'].value_counts().head(15)

print("Top 15 publisher domains:")
print(domain_counts)

In [ ]:
# Visualization of publisher analysis
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# Top publishers
publisher_counts.head(15).plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Top 15 Publishers by Article Count')
axes[0].set_xlabel('Publisher')
axes[0].set_ylabel('Number of Articles')
axes[0].tick_params(axis='x', rotation=45)

# Top domains
domain_counts.plot(kind='bar', ax=axes[1], color='darkorange')
axes[1].set_title('Top 15 Publisher Domains by Article Count')
axes[1].set_xlabel('Domain')
axes[1].set_ylabel('Number of Articles')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 3. Time Series Analysis of News Volume

In [ ]:
# Articles per year
yearly_counts = df['year'].value_counts().sort_index()
print("Articles per year:")
print(yearly_counts)

In [ ]:
# Articles per month
monthly_counts = df.groupby(['year', 'month']).size().reset_index(name='count')
monthly_counts['year_month'] = monthly_counts['year'].astype(str) + '-' + monthly_counts['month'].astype(str).str.zfill(2)

print("Monthly article counts (last 12 months):")
print(monthly_counts.tail(12))

In [ ]:
# Publishing time analysis
hourly_counts = df['hour'].value_counts().sort_index()
daily_counts = df['day_of_week'].value_counts()

print("Articles by hour of day:")
print(hourly_counts)
print("\nArticles by day of week:")
print(daily_counts)

In [ ]:
# Time series visualizations
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Articles per year
yearly_counts.plot(kind='bar', ax=axes[0,0], color='forestgreen')
axes[0,0].set_title('Articles Published per Year')
axes[0,0].set_xlabel('Year')
axes[0,0].set_ylabel('Number of Articles')

# Monthly trend
monthly_counts.set_index('year_month')['count'].plot(ax=axes[0,1], color='crimson')
axes[0,1].set_title('Monthly Article Publication Trend')
axes[0,1].set_xlabel('Year-Month')
axes[0,1].set_ylabel('Number of Articles')
axes[0,1].tick_params(axis='x', rotation=45)

# Hourly distribution
hourly_counts.plot(kind='bar', ax=axes[1,0], color='royalblue')
axes[1,0].set_title('Articles Published by Hour of Day')
axes[1,0].set_xlabel('Hour')
axes[1,0].set_ylabel('Number of Articles')

# Day of week distribution
daily_counts.plot(kind='bar', ax=axes[1,1], color='purple')
axes[1,1].set_title('Articles Published by Day of Week')
axes[1,1].set_xlabel('Day of Week')
axes[1,1].set_ylabel('Number of Articles')
axes[1,1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 4. Text Analysis and Topic Modeling

In [ ]:
# Text preprocessing for analysis
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_headline'] = df['headline'].apply(clean_text)

# Extract common words
all_words = ' '.join(df['clean_headline']).split()
word_freq = Counter(all_words)

# Remove common stop words
stop_words = {'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for', 'of', 'with', 'by', 'is', 'are', 'was', 'were', 'be', 'been', 'have', 'has', 'had', 'do', 'does', 'did', 'will', 'would', 'could', 'should', 'may', 'might', 'can', 'this', 'that', 'these', 'those', 'it', 'its', 'they', 'their', 'you', 'your', 'we', 'our', 'us', 'i', 'me', 'my', 'mine', 'he', 'him', 'his', 'she', 'her', 'hers'}

filtered_words = [word for word in all_words if word not in stop_words and len(word) > 2]
filtered_word_freq = Counter(filtered_words)

print("Top 20 most common words (excluding stop words):")
print(filtered_word_freq.most_common(20))

In [ ]:
# Identify financial keywords and phrases
financial_keywords = [
    'stock', 'shares', 'price', 'market', 'trading', 'earnings', 'revenue',
    'profit', 'loss', 'growth', 'dividend', 'yield', 'volatility', 'bull', 'bear',
    'buy', 'sell', 'hold', 'upgrade', 'downgrade', 'target', 'forecast', 'guidance',
    'merger', 'acquisition', 'ipo', 'sec', 'fda', 'approval', 'patent', 'clinical'
]

financial_word_counts = {word: filtered_word_freq[word] for word in financial_keywords if word in filtered_word_freq}
financial_word_counts = dict(sorted(financial_word_counts.items(), key=lambda x: x[1], reverse=True))

print("Financial keywords frequency:")
print(financial_word_counts)

In [ ]:
# Create word cloud
wordcloud = WordCloud(width=800, height=400, background_color='white', 
                      max_words=100, colormap='viridis').generate_from_frequencies(filtered_word_freq)

plt.figure(figsize=(12, 6))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud of Financial News Headlines', fontsize=16)
plt.show()

In [ ]:
# Text analysis visualizations
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Top 20 common words
top_words = dict(filtered_word_freq.most_common(20))
axes[0,0].barh(list(top_words.keys()), list(top_words.values()), color='teal')
axes[0,0].set_title('Top 20 Most Common Words in Headlines')
axes[0,0].set_xlabel('Frequency')
axes[0,0].invert_yaxis()

# Financial keywords
if financial_word_counts:
    axes[0,1].barh(list(financial_word_counts.keys()), list(financial_word_counts.values()), color='gold')
    axes[0,1].set_title('Financial Keywords Frequency')
    axes[0,1].set_xlabel('Frequency')
    axes[0,1].invert_yaxis()
else:
    axes[0,1].text(0.5, 0.5, 'No financial keywords found', ha='center', va='center', transform=axes[0,1].transAxes)
    axes[0,1].set_title('Financial Keywords Frequency')

# Stock symbol distribution
stock_counts = df['stock'].value_counts().head(15)
axes[1,0].barh(range(len(stock_counts)), stock_counts.values, color='coral')
axes[1,0].set_yticks(range(len(stock_counts)))
axes[1,0].set_yticklabels(stock_counts.index)
axes[1,0].set_title('Top 15 Most Mentioned Stock Symbols')
axes[1,0].set_xlabel('Number of Mentions')
axes[1,0].invert_yaxis()

# Headline length over time
daily_avg_length = df.groupby(df['date'].dt.date)['headline_length'].mean()
axes[1,1].plot(daily_avg_length.index, daily_avg_length.values, color='purple', alpha=0.7)
axes[1,1].set_title('Average Headline Length Over Time')
axes[1,1].set_xlabel('Date')
axes[1,1].set_ylabel('Average Character Count')
axes[1,1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 5. Key Insights Summary

In [ ]:
# Generate summary statistics
summary_stats = {
    'Total Articles': len(df),
    'Unique Publishers': df['publisher'].nunique(),
    'Unique Stock Symbols': df['stock'].nunique(),
    'Date Range': f"{df['date'].min().date()} to {df['date'].max().date()}",
    'Avg Headline Length': f"{df['headline_length'].mean():.1f} characters",
    'Avg Word Count': f"{df['headline_word_count'].mean():.1f} words",
    'Most Active Publisher': df['publisher'].value_counts().index[0],
    'Most Mentioned Stock': df['stock'].value_counts().index[0],
    'Peak Publishing Hour': df['hour'].value_counts().index[0],
    'Peak Publishing Day': df['day_of_week'].value_counts().index[0]
}

print("EDA Summary Statistics:")
for key, value in summary_stats.items():
    print(f"{key}: {value}")

### Key Findings:

1. **Text Characteristics**: Headlines have an average length of X characters with Y words on average
2. **Publisher Distribution**: The dataset contains articles from Z unique publishers, with [top publisher] being most active
3. **Temporal Patterns**: News volume shows peaks during [specific times/days]
4. **Content Themes**: Common financial terms include [top keywords]
5. **Stock Coverage**: The dataset covers [number] unique stock symbols, with [top stock] most frequently mentioned

This analysis provides a foundation for sentiment analysis and correlation studies in subsequent tasks.